# Exercise 3.3: Table Modeling and Write Optimization

Building on the partitioning and sort order concepts from E1.2, this exercise focuses on write-time optimization — specifically how Spark's distribution modes affect file layout and write performance.

In this exercise, you'll learn how to optimize Apache Iceberg tables using the NYC Yellow Taxi dataset:
- **Partition strategies**: Compare different partition granularities (revisited from E1.2 with a performance focus)
- **Sort orders**: Optimize for specific query patterns (revisited with write-cost analysis)
- **Distribution modes**: Control how Spark distributes data before writing (new in this exercise)

## Learning Objectives
- Compare unpartitioned, daily, and monthly partitioning
- Implement and test sort orders
- Understand distribution mode tradeoffs
- Measure the impact of modeling decisions on query performance

> **Note on benchmarks:** The configurations in this notebook were chosen to illustrate real-world tradeoffs between different modeling strategies. However, absolute timings will vary based on your local environment (CPU, memory, disk speed, Docker resource allocation). In production with distributed clusters and larger datasets, the performance differences are typically more pronounced.

## Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = SparkSession.builder \
    .appName("TableModeling") \
    .getOrCreate()

print(f"Spark {spark.version} initialized!")

## Create Namespace

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.modeling")
print("Namespace 'modeling' created!")

## Download NYC Taxi Data

We'll use NYC Yellow Taxi trip data for **June and July 2023** (~100 MB total) to compare different table modeling strategies. A larger dataset makes performance differences between strategies more visible.

In [ ]:
import boto3
from botocore.client import Config
import urllib.request
import os

s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='password',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
bucket = "warehouse"

for month in [6, 7]:
    filename = f"yellow_tripdata_2023-{month:02d}.parquet"
    key = f"raw/{filename}"
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        print(f"{filename} already in MinIO -- skipping download")
    except:
        local_path = f"/tmp/{filename}"
        print(f"Downloading {filename} (~50MB)...")
        urllib.request.urlretrieve(base_url.format(month), local_path)
        s3_client.upload_file(local_path, bucket, key)
        os.remove(local_path)
        print(f"  Uploaded to s3a://{bucket}/{key}")

print("\nTaxi data ready in MinIO!")

In [ ]:
taxi_df = spark.read.parquet(
    "s3a://warehouse/raw/yellow_tripdata_2023-06.parquet",
    "s3a://warehouse/raw/yellow_tripdata_2023-07.parquet"
)
taxi_df.cache()
taxi_df.createOrReplaceTempView("taxi_raw")
count = taxi_df.count()
print(f"Loaded {count:,} taxi trips (~100 MB) for testing")
taxi_df.show(5)

## Experiment 1: Partition Granularity

### Strategy 1: Unpartitioned Table

In [ ]:
print("Creating UNPARTITIONED table...")
start = time.time()

taxi_df.writeTo("polaris.modeling.taxi_unpartitioned") \
    .using("iceberg") \
    .tableProperty("write.target-file-size-bytes", "134217728") \
    .createOrReplace()

write_time_unpart = time.time() - start
print(f"Write time: {write_time_unpart:.2f} seconds")

In [ ]:
print("Unpartitioned file statistics:")
spark.sql("""
    SELECT COUNT(*) as file_count,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb
    FROM polaris.modeling.taxi_unpartitioned.files
""").show()

### Strategy 2: Daily Partitioning

In [ ]:
print("Creating DAILY PARTITIONED table...")
start = time.time()

spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_daily")
spark.sql("""
    CREATE TABLE polaris.modeling.taxi_daily
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES ('write.target-file-size-bytes' = '134217728')
    AS SELECT * FROM taxi_raw
""")

write_time_daily = time.time() - start
print(f"Write time: {write_time_daily:.2f} seconds")

In [ ]:
print("Daily partition file statistics:")
spark.sql("""
    SELECT COUNT(*) as file_count,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
           COUNT(DISTINCT partition) as partition_count
    FROM polaris.modeling.taxi_daily.files
""").show()

### Strategy 3: Monthly Partitioning

In [ ]:
print("Creating MONTHLY PARTITIONED table...")
start = time.time()

spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_monthly")
spark.sql("""
    CREATE TABLE polaris.modeling.taxi_monthly
    USING iceberg
    PARTITIONED BY (months(tpep_pickup_datetime))
    TBLPROPERTIES ('write.target-file-size-bytes' = '134217728')
    AS SELECT * FROM taxi_raw
""")

write_time_monthly = time.time() - start
print(f"Write time: {write_time_monthly:.2f} seconds")

In [ ]:
print("Monthly partition file statistics:")
spark.sql("""
    SELECT COUNT(*) as file_count,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
           COUNT(DISTINCT partition) as partition_count
    FROM polaris.modeling.taxi_monthly.files
""").show()

### Compare Query Performance

In [ ]:
query = """
    SELECT COUNT(*) as count, ROUND(AVG(fare_amount), 2) as avg_fare
    FROM {table}
    WHERE tpep_pickup_datetime >= TIMESTAMP '2023-06-15 00:00:00'
      AND tpep_pickup_datetime < TIMESTAMP '2023-06-16 00:00:00'
"""

print("Query: All trips on June 15, 2023")
print("=" * 60)

for name, table in [("Unpartitioned", "polaris.modeling.taxi_unpartitioned"),
                     ("Daily", "polaris.modeling.taxi_daily"),
                     ("Monthly", "polaris.modeling.taxi_monthly")]:
    start = time.time()
    result = spark.sql(query.format(table=table)).collect()
    elapsed = time.time() - start
    print(f"{name:15s}: {elapsed:.3f}s  (count: {result[0]['count']:,}, avg_fare: {result[0]['avg_fare']})")

In [ ]:
query2 = """
    SELECT COUNT(*) as count
    FROM {table}
    WHERE tpep_pickup_datetime >= TIMESTAMP '2023-06-15 14:00:00'
      AND tpep_pickup_datetime < TIMESTAMP '2023-06-15 15:00:00'
"""

print("\nQuery: Trips on June 15, 2-3 PM")
print("=" * 60)

for name, table in [("Unpartitioned", "polaris.modeling.taxi_unpartitioned"),
                     ("Daily", "polaris.modeling.taxi_daily"),
                     ("Monthly", "polaris.modeling.taxi_monthly")]:
    start = time.time()
    count = spark.sql(query2.format(table=table)).collect()[0]['count']
    elapsed = time.time() - start
    print(f"{name:15s}: {elapsed:.3f}s  (count: {count:,})")

### Partition Granularity Summary

In [ ]:
print("=" * 60)
print("PARTITION STRATEGY COMPARISON")
print("=" * 60)
print(f"{'Strategy':<20} {'Write Time':<12}")
print("-" * 40)
print(f"{'Unpartitioned':<20} {write_time_unpart:>10.2f}s")
print(f"{'Daily':<20} {write_time_daily:>10.2f}s")
print(f"{'Monthly':<20} {write_time_monthly:>10.2f}s")
print("=" * 60)

**Observations:**
- **Monthly:** Fewest files, but day-range queries must scan full month files
- **Daily:** Good balance — one file per day, efficient for day-range queries
- **Unpartitioned:** Simplest, but scans all data for any time filter

### Try It: Test with a Different Query Pattern

The benchmarks above used a single-day filter. Try a broader query (e.g., full-month scan) or a narrower one (e.g., a single hour). How do the partition strategies compare when the query aligns perfectly with the partition granularity vs when it doesn't?

In [ ]:
# my_query = """
#     SELECT COUNT(*), ROUND(AVG(total_amount), 2)
#     FROM {table}
#     WHERE tpep_pickup_datetime >= TIMESTAMP '2023-06-15 14:00:00'
#       AND tpep_pickup_datetime <  TIMESTAMP '2023-06-15 15:00:00'
# """
#
# for name, table in [("Unpartitioned", "polaris.modeling.taxi_unpartitioned"),
#                      ("Daily", "polaris.modeling.taxi_daily"),
#                      ("Monthly", "polaris.modeling.taxi_monthly")]:
#     start = time.time()
#     spark.sql(my_query.format(table=table)).collect()
#     elapsed = time.time() - start
#     print(f"{name:<20} {elapsed:.3f}s")

In [ ]:
for t in ['taxi_unpartitioned', 'taxi_daily', 'taxi_monthly']:
    spark.sql(f"DROP TABLE IF EXISTS polaris.modeling.{t}")
print("Experiment 1 tables cleaned up!")

## Experiment 2: Sort Orders

### Table Without Sort Order

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_unsorted")
spark.sql("""
    CREATE TABLE polaris.modeling.taxi_unsorted
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    AS SELECT * FROM taxi_raw
""")

print("Unsorted table created!")

### Table With Sort Order

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_sorted")
spark.sql("""
    CREATE TABLE polaris.modeling.taxi_sorted
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    AS SELECT * FROM taxi_raw WHERE 1=0
""")

spark.sql("""
    ALTER TABLE polaris.modeling.taxi_sorted
    WRITE ORDERED BY PULocationID
""")

taxi_df.writeTo("polaris.modeling.taxi_sorted").append()

print("Sorted table created (sorted by PULocationID)!")

### Compare Query Performance

In [ ]:
query = """
    SELECT COUNT(*) as count, ROUND(AVG(fare_amount), 2) as avg_fare
    FROM {table}
    WHERE PULocationID = 132
      AND tpep_pickup_datetime >= '2023-06-15' AND tpep_pickup_datetime < '2023-06-16'
"""

print("Query: Trips from location 132 on June 15")
print("=" * 60)

start = time.time()
r1 = spark.sql(query.format(table="polaris.modeling.taxi_unsorted")).collect()
unsorted_time = time.time() - start
print(f"Unsorted: {unsorted_time:.3f}s  (count: {r1[0]['count']}, avg_fare: {r1[0]['avg_fare']})")

start = time.time()
r2 = spark.sql(query.format(table="polaris.modeling.taxi_sorted")).collect()
sorted_time = time.time() - start
print(f"Sorted:   {sorted_time:.3f}s  (count: {r2[0]['count']}, avg_fare: {r2[0]['avg_fare']})")

if unsorted_time > sorted_time:
    print(f"\nSort order gave {unsorted_time/sorted_time:.1f}x speedup on filtered query")
else:
    print(f"\nResults similar at this data volume")

### Try It: Try a Different Sort Key

The sorted table above uses `PULocationID`. Try creating a table sorted by `fare_amount` or `trip_distance` instead. Run a query that filters on your sort key and compare it to the unsorted version. Does the sort order help more for high-selectivity filters?

In [ ]:
# my_sort_key = "fare_amount"  # or trip_distance, DOLocationID, etc.
# my_filter = "fare_amount > 100"  # match your sort key for best effect

# spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_my_sort")
# spark.sql("""
#     CREATE TABLE polaris.modeling.taxi_my_sort
#     USING iceberg
#     PARTITIONED BY (days(tpep_pickup_datetime))
#     AS SELECT * FROM taxi_raw
#     WHERE 1=0
# """)
# spark.sql(f"ALTER TABLE polaris.modeling.taxi_my_sort WRITE ORDERED BY {my_sort_key}")
# taxi_df.writeTo("polaris.modeling.taxi_my_sort").append()
#
# start = time.time()
# spark.sql(f"SELECT COUNT(*) FROM polaris.modeling.taxi_my_sort WHERE {my_filter}").collect()
# print(f"Sorted by {my_sort_key}: {time.time() - start:.3f}s")
#
# spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_my_sort")

In [ ]:
for t in ['taxi_unsorted', 'taxi_sorted']:
    spark.sql(f"DROP TABLE IF EXISTS polaris.modeling.{t}")
print("Experiment 2 tables cleaned up!")

## Experiment 3: Distribution Modes

Distribution mode is a Spark-specific Iceberg setting that controls whether and how Spark rearranges (shuffles) data across its worker tasks before writing. This affects how many files are produced and how well they align with the table's partition scheme. The **hash** mode groups data by partition value at the cost of a shuffle, while **none** skips the shuffle entirely and lets each task write directly — faster if the data is already organized, but risky for small file generation if it isn't.

### Hash Distribution (Default)

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_dist_hash")

print("Writing with HASH distribution mode...")
start = time.time()

spark.sql("""
    CREATE TABLE polaris.modeling.taxi_dist_hash
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES ('write.distribution-mode' = 'hash')
    AS SELECT * FROM taxi_raw
""")

hash_time = time.time() - start
print(f"Write time: {hash_time:.2f} seconds")

In [ ]:
print("Hash distribution file stats:")
spark.sql("""
    SELECT COUNT(*) as total_files, COUNT(DISTINCT partition) as partitions,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb
    FROM polaris.modeling.taxi_dist_hash.files
""").show()

### None Distribution (No Shuffle)

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_dist_none")

print("Writing with NONE distribution mode...")
start = time.time()

spark.sql("""
    CREATE TABLE polaris.modeling.taxi_dist_none
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES ('write.distribution-mode' = 'none')
    AS SELECT * FROM taxi_raw
""")

none_time = time.time() - start
print(f"Write time: {none_time:.2f} seconds")

In [ ]:
print("None distribution file stats:")
spark.sql("""
    SELECT COUNT(*) as total_files, COUNT(DISTINCT partition) as partitions,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
           ROUND(MIN(file_size_in_bytes) / 1024 / 1024, 2) as min_size_mb
    FROM polaris.modeling.taxi_dist_none.files
""").show()

### Distribution Mode Summary

In [ ]:
print("=" * 60)
print("DISTRIBUTION MODE COMPARISON")
print("=" * 60)
print(f"{'Mode':<15} {'Write Time':<12} {'Notes':<30}")
print("-" * 60)
print(f"{'Hash (default)':<15} {hash_time:>10.2f}s  {'Fewer, larger files':<30}")
print(f"{'None':<15} {none_time:>10.2f}s  {'Many small files':<30}")
print("=" * 60)

- **Hash:** Shuffle ensures one file per partition — clean, uniform files
- **None:** No shuffle, but tasks may write to many partitions — can produce many small files

Why was "none" slower despite skipping the shuffle? Each Spark task writes to every partition it encounters, producing many small file writes. Many small writes can be slower than fewer large writes after a shuffle. The shuffle costs CPU time, but it consolidates writes so each task outputs to a single partition.

### Try It: Combine Strategies

Create a table that combines partitioning, sort order, and distribution mode. For example: daily partitioning + sorted by `PULocationID` + hash distribution. Write data to it, then compare query performance against the tables from the previous experiments. What combination works best for your query pattern?

In [ ]:
# spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_combined")
# spark.sql("""
#     CREATE TABLE polaris.modeling.taxi_combined
#     USING iceberg
#     PARTITIONED BY (days(tpep_pickup_datetime))
#     TBLPROPERTIES ('write.distribution-mode' = 'hash')
#     AS SELECT * FROM taxi_raw
#     WHERE 1=0
# """)
# spark.sql("ALTER TABLE polaris.modeling.taxi_combined WRITE ORDERED BY PULocationID")
# taxi_df.writeTo("polaris.modeling.taxi_combined").append()
#
# start = time.time()
# spark.sql("""
#     SELECT COUNT(*), ROUND(AVG(fare_amount), 2)
#     FROM polaris.modeling.taxi_combined
#     WHERE PULocationID = 132
#       AND tpep_pickup_datetime >= '2023-06-15' AND tpep_pickup_datetime < '2023-06-16'
# """).collect()
# print(f"Combined strategy: {time.time() - start:.3f}s")
#
# spark.sql("DROP TABLE IF EXISTS polaris.modeling.taxi_combined")

## Cleanup

In [ ]:
taxi_df.unpersist()
for t in ['taxi_dist_hash', 'taxi_dist_none']:
    spark.sql(f"DROP TABLE IF EXISTS polaris.modeling.{t}")
print("Cleanup complete!")